In [1]:
%pip install -U transformers datasets accelerate bitsandbytes peft trl tensorboard matplotlib pandas pyyaml loguru tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
print(os.getcwd())

c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning


In [3]:
from pathlib import Path

required_paths = [
    "data/processed/train.jsonl",
    "configs/model_config.yaml",
    "src/finetuning/dataset_builder.py",
    "src/finetuning/trainer.py",
    "src/finetuning/inference.py"
]

for path in required_paths:
    p = Path(path)
    print(f"{'✅' if p.exists() else '❌'} {path}")

✅ data/processed/train.jsonl
✅ configs/model_config.yaml
✅ src/finetuning/dataset_builder.py
✅ src/finetuning/trainer.py
✅ src/finetuning/inference.py


In [4]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
from huggingface_hub import login

login()

In [6]:
import json
from pathlib import Path

train_path = Path("data/processed/train.jsonl")

assert train_path.exists(), "train.jsonl not found."

with open(train_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("Training examples:", len(rows))
print(rows[0])

Training examples: 4077
{'instruction': "Answer the question using Uber's 2024 Annual Report.", 'input': "Rewrite the description of Uber's purchase commitments in a clear and concise investor-friendly summary.", 'output': 'Uber has non-cancelable purchase commitments primarily related to network and cloud services essential to its operations. Notably, in November 2022, Uber entered into commercial technology agreements committing to spend at least $2.5 billion on cloud computing services through November 2029, with $412 million due in the short term. These commitments reflect Uber’s strategic investment in critical infrastructure to support its business growth.', 'source_chunk_id': 'uber_2024_p68_c289', 'page_reference': 'page 68', 'category': 'stylistic_creative'}


In [7]:
from src.finetuning.dataset_builder import build_sft_dataset

dataset = build_sft_dataset("data/processed/train.jsonl")

print(dataset)
print(dataset[0]["text"][:1500])

Dataset({
    features: ['text', 'source_chunk_id', 'page_reference', 'category'],
    num_rows: 4077
})
<s>[INST] <<SYS>>
You are The Intern, a fine-tuned assistant trained on Uber's 2024 Annual Report.
Answer in a clear, professional, investor-report style.
If the question asks for exact financial numbers and you are unsure, say that the number should be verified against the source report.
Do not invent figures.
<</SYS>>

Answer the question using Uber's 2024 Annual Report.

Question:
Rewrite the description of Uber's purchase commitments in a clear and concise investor-friendly summary. [/INST]
Uber has non-cancelable purchase commitments primarily related to network and cloud services essential to its operations. Notably, in November 2022, Uber entered into commercial technology agreements committing to spend at least $2.5 billion on cloud computing services through November 2029, with $412 million due in the short term. These commitments reflect Uber’s strategic investment in crit

In [9]:
from src.finetuning.trainer import train_intern

train_intern("configs/model_config.yaml")

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

c:\Users\Dunith Munasinghe\Desktop\ledger-mind\Annual-Report-Reading-RAG-Finetunning\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dunith Munasinghe\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
print("\n🖕🖕🖕🖕🖕")


🖕🖕🖕🖕🖕
